# Faz 4: BERTurk fine-tuning (Colab, T4)

Bu notebook yalnız paketin CLI'ını çağırır (`python -m ttc4900.finetune ...`). Ayarlar
`src/ttc4900/finetune.py` içindeki dondurulmuş `FT_CONFIG` / `RULES`'tadır (Faz 4a); burada hiçbir
hiperparametre yoktur. Protokol: `docs/faz4_protokol.md`.

Akış:
1. GPU kontrolü, repo klonu ve **Faz 4b commit'ine checkout**.
2. Kurulum: **paket yükseltmesi yapılmaz.** `requirements.txt`'teki (tek kaynak) eğitim paketleri
   Colab'da kuruluysa olduğu gibi kullanılır; yalnız eksik olan kurulur (pip'in kurulu bir
   paketi değiştirmeyeceği önce `--dry-run` ile doğrulanır). `torch` hiç kurulmaz. Sürümler
   yazdırılır ve `phase4_colab_env_<ts>.json` olarak metrik dizinine kaydedilir.
3. `torch.cuda.is_available()` kapısı: GPU yoksa eğitim başlamaz. Ardından sürüm koruma testleri
   (gradyan birikimi, TrainingArguments, E_final callback'i) bu ortamda koşulur; başarısızsa durur.
   Sonra veri + fold doğrulaması (`finetune verify`, eğitimsiz): Colab'da `make_cv` çalıştırılmaz.
   Dış fold'lar char_lr npz'indeki `fold_id[0]`'dan, iç validation parçaları yerelde üretilip
   commit'lenen `outputs/splits/phase4_folds.json`'dan okunur (StratifiedGroupKFold fold'ları
   sklearn < 1.8 ile farklıdır). Train'in satır sırası/etiket/metin parmak izi dosyadakiyle
   aynı değilse eğitim başlamaz.
4. Drive: metrikler Drive'a yazılır; oturum koparsa aynı hücre yeniden çalıştırılır, json'u
   yazılmış koşular atlanır. Checkpoint'ler `/content/ckpt` altında geçicidir, koşu bitince silinir.
5. Önce 1 fold x 1 seed hız ölçümü, sonra `cv` (5 fold x 3 seed), sonra `final` (3 seed).
6. Metrikler zip olarak indirilir. **GitHub'a push yok.** `summarize` ve commit yerelde yapılır.

Veri yalnız `load_train()` ile okunur; test seti Faz 4'te kullanılmaz.

In [ ]:
# 1) GPU kontrolü (Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2) Parametreler: FAZ4B_SHA'yı Faz 4b commit'inin tam SHA'sıyla doldurun.
REPO_URL = "https://github.com/furkankumrudev/ttc4900-proje.git"
FAZ4B_SHA = ""  # <- doldurun (git rev-parse HEAD, Faz 4b commit'i)
REPO = "/content/ttc4900-proje"
DRIVE_ROOT = "/content/drive/MyDrive/ttc4900/faz4"
CKPT_DIR = "/content/ckpt"  # geçici disk; Drive'a yazılmaz

assert len(FAZ4B_SHA) >= 7, "FAZ4B_SHA boş: Faz 4b commit SHA'sını yazın."

In [ ]:
# 3) Klon + Faz 4b commit'ine checkout (her adım başarısızsa hücre hata verir)
import os
import subprocess


def run(*cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:  # git'in kendi hata mesajı görünsün
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(f"{' '.join(cmd)} başarısız (returncode {r.returncode})")
    return r.stdout.strip()


if not os.path.exists(REPO):
    run("git", "clone", "-q", REPO_URL, REPO)
os.chdir(REPO)
run("git", "fetch", "-q", "origin")
run("git", "checkout", "-q", FAZ4B_SHA)
HEAD = run("git", "rev-parse", "HEAD")
assert HEAD.startswith(FAZ4B_SHA), f"HEAD {HEAD} Faz 4b commit'i {FAZ4B_SHA} değil"
print(run("git", "log", "-1", "--oneline"))
dirty = run("git", "status", "--porcelain")
assert not dirty, f"çalışma ağacı temiz değil (Faz 4b commit'iyle birebir değil):\n{dirty}"
print("çalışma ağacı temiz")

In [ ]:
# 4) Kurulum: PAKET YÜKSELTMESİ YOK. requirements.txt'teki (tek kaynak) eğitim paketleri Colab'da
#    kuruluysa olduğu gibi kullanılır; yalnız EKSİK olan kurulur ve pip'in kurulu hiçbir paketi
#    değiştirmeyeceği önce --dry-run ile doğrulanır. torch hiç kurulmaz (CUDA'lı sürüm korunur).
import json
import platform
import re
import sys
import time
from importlib import metadata

try:
    from packaging.requirements import Requirement
except ImportError:  # packaging yoksa pip'in kendi kopyası
    from pip._vendor.packaging.requirements import Requirement


def norm(name):
    return re.sub(r"[-_.]+", "-", name).lower()


def installed():
    return {norm(d.metadata["Name"] or ""): d.version for d in metadata.distributions()}


NEEDED = {"datasets", "transformers", "accelerate", "scikit-learn", "scipy", "numpy", "pandas",
          "pytest"}
reqs = {}
for line in open("requirements.txt", encoding="utf-8"):
    spec = line.split("#")[0].strip()
    if spec and norm(Requirement(spec).name) in NEEDED:
        reqs[norm(Requirement(spec).name)] = Requirement(spec)
assert set(reqs) == NEEDED, f"requirements.txt'te eksik: {NEEDED - set(reqs)}"
assert "torch" not in reqs

before = installed()
missing = sorted(str(r) for n, r in reqs.items() if n not in before)
too_old = [f"{n} {before[n]} ({r})" for n, r in sorted(reqs.items())
           if n in before and not r.specifier.contains(before[n], prereleases=True)]
assert not too_old, f"Kurulu sürüm alt sınırın altında; yükseltme yapılmaz, durun: {too_old}"
if missing:
    rep = "/tmp/pip_report.json"
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "-q", "--report", rep,
                        *missing])
    assert r.returncode == 0, "pip --dry-run başarısız"
    plan = {norm(i["metadata"]["name"]): i["metadata"]["version"]
            for i in json.load(open(rep, encoding="utf-8"))["install"]}
    touched = {n: (before[n], v) for n, v in plan.items() if n in before}
    assert not touched, f"pip kurulu paketleri değiştirecekti (yükseltme yok): {touched}"
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing])
    assert r.returncode == 0, "pip kurulumu başarısız: eğitim başlamaz."
after = installed()
changed = {n: (v, after.get(n)) for n, v in before.items() if after.get(n) != v}
assert not changed, f"Kurulu paketler değişti (olmamalı): {changed}"

PKGS = ["torch", "transformers", "tokenizers", "safetensors", "huggingface-hub", "accelerate",
        "datasets", "pyarrow", "scikit-learn", "scipy", "numpy", "pandas", "pytest", "packaging"]
COLAB_ENV = {
    "time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repo_head": HEAD,
    "python": platform.python_version(),
    "packages": {p: after.get(p) for p in PKGS},
    "installed_missing": missing,
    "upgraded": [],  # yükseltme yapılmadı (yukarıdaki assert'ler)
}
with open("/content/phase4_colab_env.json", "w", encoding="utf-8") as f:
    json.dump(COLAB_ENV, f, indent=2)
print(json.dumps(COLAB_ENV, indent=2))

In [ ]:
# 5) Yol + GPU kapısı. Çalışma zamanı yeniden başlarsa (pip "restart" isterse) 2. hücreden
#    itibaren sırayla yeniden çalıştırın: HEAD ve COLAB_ENV 3-4. hücrelerde yeniden kurulur.
import os
import sys

sys.path.insert(0, f"{REPO}/src")
os.environ["PYTHONPATH"] = f"{REPO}/src"  # !python alt süreçleri paketi bulsun

import accelerate
import datasets
import torch
import transformers

print("torch", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers", transformers.__version__, "| datasets", datasets.__version__,
      "| accelerate", accelerate.__version__)
assert torch.cuda.is_available(), "GPU yok: eğitim başlamaz (CLI da --smoke dışında çıkar)."
print("GPU:", torch.cuda.get_device_name(0))

COLAB_ENV |= {"cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)}
with open("/content/phase4_colab_env.json", "w", encoding="utf-8") as f:
    json.dump(COLAB_ENV, f, indent=2)

In [ ]:
# 5b) Sürüm koruma testleri (bu Colab ortamındaki transformers/torch ile): 8 x 2 birikim =
#     batch 16 ortalaması, TrainingArguments = FT_CONFIG, E_final'da durdurma. Başarısızsa durur.
r = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_finetune.py",
     "-k", "gradient_accumulation or training_args or stop_at_epoch"],
    capture_output=True, text=True,
)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "Sürüm koruma testleri başarısız: eğitim başlamaz."

In [ ]:
# 5c) Veri + fold doğrulaması (eğitim yok, make_cv ÇALIŞTIRILMAZ): load_train() parmak izi (row_id
#     sırası, etiketler, metin) = outputs/splits/phase4_folds.json; dış fold'lar = char_lr npz
#     fold_id[0]; iç validation = fold dosyası; gruplar ayrık. Uyuşmazsa eğitim başlamaz
#     (cv/final de aynı denetimi eğitimden önce yapar).
r = subprocess.run([sys.executable, "-m", "ttc4900.finetune", "verify"], capture_output=True,
                   text=True)
print(r.stdout[-4000:], r.stderr[-3000:])
assert r.returncode == 0, "Veri/fold doğrulaması başarısız: eğitim başlamaz."

In [ ]:
# 6) Drive: metrikler ve final modeller Drive'da (git dışı)
from google.colab import drive

drive.mount("/content/drive")
OUT_DIR = f"{DRIVE_ROOT}/metrics"
MODEL_DIR = f"{DRIVE_ROOT}/models"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(OUT_DIR, MODEL_DIR, CKPT_DIR)

# kurulum sonrası sürümler (hücre 4-5) metriklerle birlikte saklanır
env_path = f"{OUT_DIR}/phase4_colab_env_{time.strftime('%Y%m%d-%H%M%S')}.json"
with open(env_path, "w", encoding="utf-8") as f:
    json.dump(COLAB_ENV, f, indent=2)
print(env_path)

## Hız ölçümü (önerilir)

Aynı dondurulmuş ayarla tek koşu: fold 0, seed 42. Bu **normal bir CV koşusudur**: json'u yazılır,
sonraki `cv` hücresinde atlanır. Süreyi (çıktıdaki `sn`) 15 ile çarparak toplam CV süresini
kestirin (çöken koşu olursa her biri için bir koşu daha eklenir). `docs/faz4_protokol.md`'deki
T4 tahmini (CV ~2.5-3.5 saat, final ~30 dk) bu ölçümle yerelde güncellenir.

In [ ]:
%%time
!python -m ttc4900.finetune cv --folds 0 --seeds 42 --out-dir "{OUT_DIR}" --ckpt-dir "{CKPT_DIR}"

## CV: 5 fold x 3 seed

Oturum koparsa 2-6. hücreleri yeniden çalıştırıp bu hücreyi tekrar çalıştırın: json'u yazılmış
koşular atlanır. Çöken koşu (best val macro-F1 < 0.5) seed + 100 ile bir kez otomatik yeniden
koşulur (`RULES["collapse"]`).

In [ ]:
%%time
!python -m ttc4900.finetune cv --out-dir "{OUT_DIR}" --ckpt-dir "{CKPT_DIR}"

## Final modeller (3 seed, tüm train, validation yok)

E_final = 15 CV koşusunun best epoch medyanı (CLI hesaplar; eksik koşu varsa çıkar). Modeller
`MODEL_DIR/<seed>/` altına yazılır. Teste taşınan model seed 42'dir (önceden seçildi).
`FINAL_RULES['final_collapse']`: bir final model train üzerinde çökmüşse (in-sample macro-F1
< 0.5) aynı E_final ile seed + 100 bir kez otomatik yeniden eğitilir; seed 42 çökerse birincil
model seed 142'dir (json'larda `primary_for_test`, `primary_model_dir`).

In [ ]:
%%time
!python -m ttc4900.finetune final --out-dir "{OUT_DIR}" --ckpt-dir "{CKPT_DIR}" --model-dir "{MODEL_DIR}"

In [ ]:
# Metrikleri zip'le (Drive'a da kopyalanır) ve indir. GitHub'a push yok.
import shutil
import time

from google.colab import files

stamp = time.strftime("%Y%m%d-%H%M%S")
zip_path = shutil.make_archive(f"/content/faz4_metrics_{stamp}", "zip", OUT_DIR)
shutil.copy(zip_path, DRIVE_ROOT)
print(zip_path)
files.download(zip_path)

## Yerelde (Colab'da değil)

1. Zip'i `outputs/metrics/` içine açın (json'lar kökte, npz'ler `preds/` altında).
2. Özet (CPU, ağ gerekmez):
   `.venv\Scripts\python -m ttc4900.finetune summarize`
   -> `phase4_summary_<ts>.json` ve seed 42 OOF'u `phase4_berturk_ft_<ts>.json` + npz.
3. Birincil güç analizi: `.venv\Scripts\python -m ttc4900.paired_test --power`
4. Denetçiler (leakage-reviewer, stats-reviewer), sonra commit. Final modeller Drive'da kalır,
   git'e girmez.